# ReBRAC Broad Validation — S3.d Parity Extension

**目的**：把 5 个 representative cell 从 2-seed 扩到 5-seed，让报告里所有 axis-level claim 都建立在 5-seed parity 基础上。

| Spoke | Pair | 现状 | 用途 |
|---|---|---|---|
| **A1**       | actorb_4p0__criticb_2p0 (anchor) | 2 → 5 | 让 A1 refit_b vs anchor 是 5v5 公平比较 |
| **A2-td3bc** | alpha_0p25                       | 2 → 5 | **paper claim 关键**：5v5 检验 ReBRAC vs TD3+BC 方向 |
| **A3**       | actorb_4p0__criticb_2p0 (anchor) | 2 → 5 | 让 A3 refit_b vs anchor 是 5v5；坐实 1-seed refit decision footnote |
| **B1**       | actorb_4p0__criticb_2p0          | 2 → 5 | sensor envelope claim 是 B 轴整轴 takeaway |
| **B2**       | actorb_4p0__criticb_2p0          | 2 → 5 | 同上 |

**新跑 seeds**：[43, 45, 46] × 5 cell = 15 runs，估 ~7.5h L4，单 Colab session 完成。

**输出位置**：与现有结果同目录 `results/offline/rebrac/broad_validation/<spoke>/<pair>/test/seed_*.json`，summarize 自动合并。

**驱动机制**：`scripts/run_offline_rebrac_broad.sh` 的 skip-resume — 已有 seed [42, 44] 跳过，只跑 [43, 45, 46]。

## 0. GPU sanity

In [ ]:
!nvidia-smi | head -10
import torch
print(f"torch={torch.__version__} cuda={torch.cuda.is_available()} device={torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'}")

## 1. Mount Drive + cwd

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)
%cd /content/drive/MyDrive/Colab Notebooks/new_offRL/rl_v2_5

## 2. 通用配置

In [ ]:
import os
import json
from pathlib import Path

# Anchor (Stage C, crosscomp/s0/cross_stream, 5 seeds)
ANCHOR_MEAN, ANCHOR_STD = 0.902, 0.021

RESULTS_ROOT = Path("results/offline/rebrac/broad_validation")
SUMMARIES_DIR = RESULTS_ROOT / "summaries"

# 5 个目标 cell：(spoke_id, pair_tag) — pair_tag 仅供 preflight/打印用，
# 实际 driver 用 spoke_id 通过 registry 查 anchor 默认值
PARITY_CELLS = [
    ("A1",       "actorb_4p0__criticb_2p0"),  # anchor
    ("A2-td3bc", "alpha_0p25"),               # td3bc α=0.25
    ("A3",       "actorb_4p0__criticb_2p0"),  # anchor
    ("B1",       "actorb_4p0__criticb_2p0"),  # anchor (B1 默认即 anchor)
    ("B2",       "actorb_4p0__criticb_2p0"),  # anchor (B2 默认即 anchor)
]

EXISTING_SEEDS = [42, 44]
TARGET_SEEDS   = [42, 43, 44, 45, 46]
NEW_SEEDS      = [s for s in TARGET_SEEDS if s not in EXISTING_SEEDS]
print(f"existing seeds: {EXISTING_SEEDS}  →  new seeds to run: {NEW_SEEDS}")

## 3. Preflight — 确认 2-seed 已存在 + 打印当前数字

如果有任何 cell 的 [42, 44] 不存在，**停止**。说明环境路径或 driver 状态有问题，强行跑会浪费 7.5h 并污染结果。

In [ ]:
rows = []
missing = []

for spoke_id, pair in PARITY_CELLS:
    test_dir = RESULTS_ROOT / spoke_id / pair / "test"
    succs = []
    for s in EXISTING_SEEDS:
        path = test_dir / f"seed_{s}.json"
        if not path.exists():
            missing.append(str(path))
            continue
        d = json.loads(path.read_text(encoding="utf-8"))
        succs.append(d["eval_success_rate"])
    if len(succs) == len(EXISTING_SEEDS):
        mean = sum(succs) / len(succs)
        rows.append((spoke_id, pair, succs, mean))

print("=" * 88)
print(f"{'spoke':<10}{'pair':<32}{'seeds':>10}  {'succ@seed':<24}{'mean':>9}")
print("-" * 88)
for spoke_id, pair, succs, mean in rows:
    succ_str = ", ".join(f"{s:.3f}" for s in succs)
    print(f"{spoke_id:<10}{pair:<32}{str(EXISTING_SEEDS):>10}  {succ_str:<24}{mean:>9.4f}")
print("=" * 88)

if missing:
    print(f"\n[FAIL] missing {len(missing)} seed JSON files — DO NOT proceed:")
    for m in missing:
        print(f"  {m}")
    raise RuntimeError("preflight failed: existing P1 seeds missing for parity targets")
else:
    print(f"\n[OK] all 5 cells × {len(EXISTING_SEEDS)} seeds present → safe to extend")

## 4. 跑 parity extension — 15 runs

每个 cell 跑 SEEDS=[42 43 44 45 46]；driver skip-resume 会跳过 [42, 44]，只训 [43, 45, 46]。

**重要**：A2-td3bc 走 td3bc 分支（`--alpha 0.25`），ReBRAC penalty 系数被脚本忽略。其余 4 个走 rebrac 分支并自动从 spoke registry 读到 anchor (4.0, 2.0)，无需 env 覆盖。

In [ ]:
# 显式覆盖 SEEDS 也可以，但 PHASE=p2_5seed 默认即 "42 43 44 45 46"，留空即可。
PARITY_SPOKES = [c[0] for c in PARITY_CELLS]

for spoke_id in PARITY_SPOKES:
    os.environ["SPOKE_ID"] = spoke_id
    os.environ["PHASE"]    = "p2_5seed"
    # 不设 ACTOR_PENALTY_COEFS / CRITIC_PENALTY_COEFS / TD3BC_ALPHAS：
    # script 会从 spoke registry 读 anchor 默认值 (rebrac) 或 alpha (td3bc)
    print(f"\n========== S3.d parity: {spoke_id} (5-seed, anchor defaults) ==========")
    !bash scripts/run_offline_rebrac_broad.sh

## 5. 重跑 summarize

In [ ]:
!python -m scripts.summarize_broad_validation

## 6. Updated representative-cell 表

每个 spoke 取 `n_seeds` 最大的 cell 作为 representative。所有 5 个 parity cell 应该升级到 5-seed。

In [ ]:
p1_overview = json.loads((SUMMARIES_DIR / "p1_overview.json").read_text(encoding="utf-8"))

# 每个 spoke 取 n_seeds 最大的 cell
spoke_best = {}
for row in p1_overview:
    sid = row["spoke_id"]
    cur = spoke_best.get(sid)
    if cur is None or row["num_seeds"] > cur["num_seeds"]:
        spoke_best[sid] = row

axis_map = {
    "A1": "A", "A2": "A", "A2-td3bc": "A", "A3": "A",
    "B1": "B", "B2": "B",
    "C1": "C", "C3": "C",
}

print("=" * 100)
print(f"{'axis':<6}{'spoke':<10}{'pair':<32}{'n':>4}{'mean':>9}{'std':>9}{'Δ vs anchor':>14}{'status':>10}")
print("-" * 100)

for sid in ["A1", "A2", "A2-td3bc", "A3", "B1", "B2", "C1", "C3"]:
    row = spoke_best.get(sid)
    if row is None:
        print(f"{axis_map[sid]:<6}{sid:<10}{'MISSING':<32}")
        continue
    m = row["mean_test_success_rate"] or 0.0
    s = row["std_test_success_rate"]  or 0.0
    n = row["num_seeds"]
    status = "✅ 5-seed" if n >= 5 else f"⚠ {n}-seed"
    print(f"{axis_map[sid]:<6}{sid:<10}{row['pair']:<32}{n:>4}"
          f"{m:>9.4f}{s:>9.4f}{(m - ANCHOR_MEAN)*100:>+12.2f}pp{status:>10}")
print("=" * 100)
print(f"\nAnchor: {ANCHOR_MEAN:.3f} ± {ANCHOR_STD:.3f} (5 seeds, Stage C crosscomp/s0/cross_stream)")

## 7. A 轴 head-to-head：refit_b vs anchor at 5-seed parity

A1 / A3 在 §6.1 winner select 时基于 1-seed 选了 refit_b。扩到 5-seed 后这个决策是否仍成立？

In [ ]:
def get_cell(spoke_id, pair):
    for row in p1_overview:
        if row["spoke_id"] == spoke_id and row["pair"] == pair:
            return row
    return None

print("=" * 96)
print(f"{'spoke':<8}{'cell':<32}{'n':>4}{'mean':>9}{'std':>9}{'Δ vs anchor':>14}{'verdict':>20}")
print("-" * 96)

for spoke_id in ["A1", "A3"]:
    anchor   = get_cell(spoke_id, "actorb_4p0__criticb_2p0")
    refit_b  = get_cell(spoke_id, "actorb_4p0__criticb_1p0")
    rows = [("anchor", anchor), ("refit_b", refit_b)]
    succs = []
    for tag, row in rows:
        if row is None:
            print(f"{spoke_id:<8}{tag + ' MISSING':<32}")
            continue
        m = row["mean_test_success_rate"] or 0.0
        s = row["std_test_success_rate"]  or 0.0
        n = row["num_seeds"]
        succs.append((tag, m, n))
        print(f"{spoke_id:<8}{tag + ' (' + row['pair'] + ')':<32}{n:>4}"
              f"{m:>9.4f}{s:>9.4f}{(m - ANCHOR_MEAN)*100:>+12.2f}pp")
    if len(succs) == 2:
        a_tag, a_m, a_n = succs[0]
        b_tag, b_m, b_n = succs[1]
        diff = b_m - a_m
        winner = b_tag if b_m > a_m else a_tag
        same_n = "✓ 5v5" if a_n == b_n == 5 else f"⚠ {a_n}v{b_n}"
        print(f"{'':<8}{'  → ' + winner + f' wins by {abs(diff)*100:.2f}pp':<32}{'':<26}{same_n:>20}")
    print()
print("=" * 96)

## 8. A2 mid-gap：ReBRAC vs TD3+BC at 5-seed parity ⭐

**这是 paper rev.8 claim 的关键检验**。spec §6.1 mid-gap criterion: |ReBRAC − TD3BC| < 5pp 触发。

In [ ]:
rebrac_cell = get_cell("A2",       "actorb_4p0__criticb_2p0")
td3bc_cell  = get_cell("A2-td3bc", "alpha_0p25")

print("=" * 88)
print(f"{'algo':<14}{'cell':<32}{'n':>4}{'mean':>9}{'std':>9}{'Δ vs anchor':>14}")
print("-" * 88)

if rebrac_cell:
    m = rebrac_cell["mean_test_success_rate"] or 0.0
    s = rebrac_cell["std_test_success_rate"]  or 0.0
    n = rebrac_cell["num_seeds"]
    print(f"{'A2 ReBRAC':<14}{rebrac_cell['pair']:<32}{n:>4}{m:>9.4f}{s:>9.4f}{(m - ANCHOR_MEAN)*100:>+12.2f}pp")
    rebrac_m, rebrac_n = m, n
else:
    rebrac_m = rebrac_n = None

if td3bc_cell:
    m = td3bc_cell["mean_test_success_rate"] or 0.0
    s = td3bc_cell["std_test_success_rate"]  or 0.0
    n = td3bc_cell["num_seeds"]
    print(f"{'A2 TD3+BC':<14}{td3bc_cell['pair']:<32}{n:>4}{m:>9.4f}{s:>9.4f}{(m - ANCHOR_MEAN)*100:>+12.2f}pp")
    td3bc_m, td3bc_n = m, n
else:
    td3bc_m = td3bc_n = None

print("=" * 88)
if rebrac_m is not None and td3bc_m is not None:
    diff_pp = (rebrac_m - td3bc_m) * 100
    direction = "ReBRAC > TD3+BC" if diff_pp > 0 else ("ReBRAC < TD3+BC" if diff_pp < 0 else "tied")
    matched = "✓ matched 5v5" if rebrac_n == td3bc_n == 5 else f"⚠ unmatched {rebrac_n}v{td3bc_n}"
    print(f"\n  Δ = {diff_pp:+.2f}pp  ({direction})  [{matched}]")
    if abs(diff_pp) < 5.0:
        print(f"  spec §6.1 mid-gap: |Δ| < 5pp → A2 仍触发 (algorithm 几乎打平)")
    else:
        if diff_pp > 0:
            print(f"  spec §6.1 mid-gap: |Δ| ≥ 5pp 且方向正 → ReBRAC ≥ TD3+BC (paper claim 站住)")
        else:
            print(f"  spec §6.1 mid-gap: |Δ| ≥ 5pp 且方向负 → ⚠ TD3+BC > ReBRAC (paper claim 需要限定条件)")

## 9. 最终 verdict — 5-seed parity 后是否可以进入报告

通过 = 三条都满足：
1. 5 个 parity cell 都升到 5-seed（status 列全 ✅）
2. A1 / A3 winner 决策与 1-seed refit 决策一致 OR 已经 flagged
3. A2 ReBRAC vs TD3+BC 比较是 matched 5v5

In [ ]:
checks = []
for spoke_id, pair in PARITY_CELLS:
    row = get_cell(spoke_id, pair)
    n = row["num_seeds"] if row else 0
    checks.append((f"{spoke_id} / {pair}", n >= 5, f"n={n}"))

# A2 mid-gap check
a2_n      = rebrac_cell["num_seeds"] if rebrac_cell else 0
td3bc_n_  = td3bc_cell["num_seeds"]  if td3bc_cell  else 0
checks.append(("A2 ReBRAC vs TD3+BC matched seeds", a2_n == td3bc_n_ == 5, f"{a2_n}v{td3bc_n_}"))

print("=" * 80)
print(f"{'check':<60}{'pass':>6}{'detail':>14}")
print("-" * 80)
all_pass = True
for name, ok, detail in checks:
    mark = "✅" if ok else "❌"
    if not ok:
        all_pass = False
    print(f"{name:<60}{mark:>6}{detail:>14}")
print("=" * 80)
print(f"\n{'✅ READY for report drafting' if all_pass else '❌ HOLD — re-run failed cells before report'}")